# nanochat: Talking to the Model
## Yoav Ram

The previous three notebooks each ended by writing a checkpoint:

| Notebook | Checkpoint | What it learned |
|---|---|---|
| `nanochat.ipynb` | `checkpoints/nanochat_checkpoint.pkl` | next-token prediction on TinyStories |
| `nanochat-sft.ipynb` | `checkpoints/nanochat_sft_checkpoint.pkl` | to answer in the `[INST] … [/INST]` chat format |
| `nanochat-grpo.ipynb` | `checkpoints/nanochat_grpo_best.pkl` | to satisfy a sentence-count constraint |

This notebook is where we *use* them. We build the inference half of a language model — the part that turns trained weights into something you can talk to — and then push the same prompt through all three checkpoints to see what pretraining, SFT and GRPO each actually bought.

Along the way there are two ideas that only show up at inference time:

- **sampling** — training only ever asked for a probability distribution; generation has to *choose*, and how we choose changes the output more than most people expect;
- **shape stability in JAX** — the obvious generation loop is around 20× slower per token than it needs to be, and 90× slower to start, for a reason that has nothing to do with the model.

**Prerequisites.** This notebook trains nothing, but it needs the checkpoints above, and they are *not* distributed with the repository. Run `nanochat.ipynb`, `nanochat-sft.ipynb` and `nanochat-grpo.ipynb` first. The loading cell reports which it found and works with whatever subset exists.

**Hardware.** Runs on CPU, slowly. On a GPU, generation costs a few milliseconds per token after a one-off compile of a few seconds.

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # quieten the XLA autotuner

import math
import pickle
import re
import time

import jax
import jax.numpy as jnp
print('jax', jax.__version__, jax.default_backend())

from bpe import bpe_encode, bpe_decode

jax 0.9.2 gpu


## 1. Loading a checkpoint

A checkpoint is a pickle holding three things: the parameters, the config describing their shapes, and the tokenizer's vocabulary and merge list.

Keeping the tokenizer *inside* the checkpoint is deliberate. Token id 412 means nothing without the vocabulary that produced it, and a model loaded against the wrong tokenizer does not crash — it produces confident nonsense. Shipping them together makes that mistake hard to commit.

`load_checkpoint` returns a plain dict. Note that it converts the stored NumPy arrays back to JAX arrays and the merge list back to tuples: pickle round-trips lists, but the encoder needs tuples as dictionary keys.

In [2]:
def load_checkpoint(path):
    with open(path, 'rb') as f:
        state = pickle.load(f)
    state['params'] = jax.tree_util.tree_map(jnp.array, state['params'])
    state['merges'] = [tuple(m) for m in state['merges']]
    return state

CHECKPOINTS = {
    'pretrained': 'checkpoints/nanochat_checkpoint.pkl',
    'SFT':        'checkpoints/nanochat_sft_checkpoint.pkl',
    'GRPO':       'checkpoints/nanochat_grpo_best.pkl',
}

models = {}
for name, path in CHECKPOINTS.items():
    if not os.path.exists(path):
        print(f'MISSING   {name:11s} {path}')
        continue
    models[name] = load_checkpoint(path)
    n_params = sum(a.size for a in jax.tree_util.tree_leaves(models[name]['params']))
    print(f'loaded    {name:11s} {path}  ({n_params:,} parameters)')

if not models:
    raise FileNotFoundError(
        'No checkpoints found. Run nanochat.ipynb (then sft, then grpo) first.')

reference = next(iter(models.values()))
cfg, vocab, merges = reference['cfg'], reference['vocab'], reference['merges']
print()
print('config:', cfg)
print(f'vocabulary: {len(vocab)} tokens from {len(merges)} merges')

loaded    pretrained  checkpoints/nanochat_checkpoint.pkl  (26,223,104 parameters)
loaded    SFT         checkpoints/nanochat_sft_checkpoint.pkl  (26,223,104 parameters)
loaded    GRPO        checkpoints/nanochat_grpo_best.pkl  (26,223,104 parameters)

config: {'vocab_size': 1024, 'd_model': 512, 'n_heads': 8, 'n_layers': 8, 'd_ff': 2048, 'head_dim': 64, 'seq_len': 256}
vocabulary: 1024 tokens from 797 merges


## 2. The model

Generating text needs a forward pass, so the model definition has to be available here. The cell below is **copied verbatim from `nanochat.ipynb`**: the same RMSNorm, RoPE, QK-normalised attention and squared-ReLU MLP.

Copying it is a wart, and an honest one to point out — `nanochat-sft.ipynb` and `nanochat-grpo.ipynb` carry the same block, so this is the fourth transcription of one model in this repository. Four copies of anything drift apart, and these already have. The fix is for `nanochat.ipynb` to *emit* a `nanochat_model.py` that every downstream notebook imports; that is a later revision of this course. For now, read this cell as a recap rather than as new material.

In [3]:
def rms_norm(g, x, eps=1e-6):
    return g * x / jnp.sqrt(jnp.mean(x**2, axis=-1, keepdims=True) + eps)

def precompute_rope(seq_len, head_dim, base=10000.):
    i = jnp.arange(0, head_dim, 2)
    angles = jnp.outer(jnp.arange(seq_len), 1.0 / (base ** (i / head_dim)))
    angles = jnp.concatenate([angles, angles], axis=-1)
    return jnp.cos(angles), jnp.sin(angles)

def apply_rope(x, cos, sin):
    d = x.shape[-1] // 2
    return x * cos + jnp.concatenate([-x[..., d:], x[..., :d]], axis=-1) * sin

def causal_mask(T):
    return jnp.where(jnp.tril(jnp.ones((T, T))), 0., -jnp.inf)

def attention_forward(p, x, cos, sin, mask):
    B, T, d = x.shape
    hd = cos.shape[-1]
    H = d // hd
    Q, K, V = x @ p['Wq'], x @ p['Wk'], x @ p['Wv']
    def split_heads(t):
        return t.reshape(B, T, H, hd).transpose(0, 2, 1, 3)
    Q, K, V = split_heads(Q), split_heads(K), split_heads(V)
    c, s = cos[None, None], sin[None, None]
    # QK-norm. RoPE is a rotation, so norm(apply_rope(Q)) == norm(Q) and it does
    # not matter that the norm is taken before the rotation rather than after.
    Q = apply_rope(Q, c, s) / (jnp.linalg.norm(Q, axis=-1, keepdims=True) + 1e-6)
    K = apply_rope(K, c, s) / (jnp.linalg.norm(K, axis=-1, keepdims=True) + 1e-6)
    w = jax.nn.softmax(Q @ K.transpose(0, 1, 3, 2) / math.sqrt(hd) + mask[None, None], axis=-1)
    return (w @ V).transpose(0, 2, 1, 3).reshape(B, T, d) @ p['Wo'], w

def mlp_forward(p, x):
    return jax.nn.relu(x @ p['W1']) ** 2 @ p['W2']

def forward(params, ids, cos, sin, mask):
    x = params['tok_emb']['W'][ids]
    for blk in params['blocks']:
        x = x + attention_forward(blk['attn'], rms_norm(blk['norm1']['g'], x), cos, sin, mask)[0]
        x = x + mlp_forward(blk['mlp'], rms_norm(blk['norm2']['g'], x))
    return rms_norm(params['norm_f']['g'], x) @ params['head']['W'].T

## 3. Sampling: from logits to a token

The forward pass ends with a vector of `vocab_size` logits. Training compared that vector against the true next token and never had to pick one. Generation does, and the picking rule is a real design choice.

**Greedy** takes the argmax. It is deterministic, which is useful for debugging, and it is a poor way to write stories: a small model falls into loops, because the single most likely continuation of a repeated phrase is usually the same repeated phrase.

**Temperature** divides the logits by $T$ before the softmax. $T < 1$ sharpens the distribution toward the mode; $T > 1$ flattens it; $T \to 0$ recovers greedy. Note it acts on the *logits*, so it is a rescaling in log-space, not a reweighting of probabilities.

**Top-$k$** keeps the $k$ largest logits and sets the rest to $-\infty$. It bounds how bad a sampled token can be, but $k$ is a fixed budget applied to a distribution whose sharpness varies wildly from position to position.

**Top-$p$ (nucleus)** instead keeps the smallest set of tokens whose cumulative probability exceeds $p$. Where the model is confident that set is tiny; where it is unsure the set grows. It adapts, which is why it is usually the better default.

In [4]:
def sample_token(key, logits, temperature=1.0, top_k=0, top_p=1.0):
    if temperature == 0.0:
        return jnp.argmax(logits)                 # greedy

    logits = logits / temperature

    if top_k > 0:                                 # keep the k largest logits
        kth = jnp.sort(logits)[-top_k]
        logits = jnp.where(logits < kth, -jnp.inf, logits)

    probs = jax.nn.softmax(logits)

    if top_p < 1.0:                               # keep the nucleus
        order = jnp.argsort(-probs)
        cumulative = jnp.cumsum(probs[order])
        # always keep the top token, then every token whose predecessors have
        # not yet accumulated p of the mass
        keep = jnp.concatenate([jnp.array([True]), cumulative[:-1] < top_p])
        mask = jnp.zeros_like(probs).at[order].set(keep.astype(probs.dtype))
        probs = probs * mask
        probs = probs / probs.sum()

    return jax.random.choice(key, probs.shape[0], p=probs)

### What the distribution actually looks like

Abstractions about sharpness are easier to believe after looking at one. Below we take a single real prompt, run one forward pass, and inspect the next-token distribution before any filtering — the ten most likely tokens, and how many tokens each nucleus threshold keeps out of the full vocabulary.

In [5]:
demo = models.get('SFT', reference)
prompt = '[INST] Continue the story: Once upon a time there was a little cat named Mia. [/INST] '

ids = bpe_encode(prompt, vocab, merges)
T = len(ids)
cos_1, sin_1 = precompute_rope(T, cfg['head_dim'])
logits = forward(demo['params'], jnp.array([ids]), cos_1, sin_1, causal_mask(T))[0, -1]
probs = jax.nn.softmax(logits)
order = jnp.argsort(-probs)

print(f'{T} prompt tokens, {len(vocab)} candidates for the next one')
print()
print('rank  token          probability')
for rank in range(10):
    t = int(order[rank])
    print(f'{rank + 1:4d}  {vocab[t]!r:13s}  {float(probs[t]):.4f}')

print()
cumulative = jnp.cumsum(probs[order])
for p in (0.5, 0.9, 0.95, 0.99):
    kept = int((cumulative < p).sum()) + 1
    print(f'top-p = {p:4}  keeps {kept:4d} of {len(vocab)} tokens')

50 prompt tokens, 1024 candidates for the next one

rank  token          probability


   1  'was'          0.3670
   2  'loved'        0.2697
   3  'had'          0.0972
   4  'liked'        0.0601
   5  'lived'        0.0477
   6  'she'          0.0258
   7  'day,'         0.0149
   8  'day'          0.0139
   9  'She'          0.0092
  10  'wanted'       0.0077



top-p =  0.5  keeps    2 of 1024 tokens
top-p =  0.9  keeps    9 of 1024 tokens
top-p = 0.95  keeps   26 of 1024 tokens
top-p = 0.99  keeps  158 of 1024 tokens


## 4. Generation, and why shape stability matters in JAX

Generation is a loop: run the model, sample a token, append it, repeat.

The obvious implementation feeds the model exactly the tokens produced so far, so the sequence grows by one every step. That is correct, and it is a trap. `jax.jit` compiles a separate kernel for **every distinct input shape**, so a context that grows by one token per step triggers a fresh XLA compilation on every single token. Measured on this repository's checkpoint, generating 60 tokens that way took **429 s** on the first call — almost all of it compiling, none of it arithmetic — and about 3.4 s on later calls, once kernels for those particular lengths had been cached. That steady state is still ~57 ms/token.

The fix is to stop changing the shape. Allocate one buffer of `seq_len` tokens, write the prompt into the front of it, and let the model see the whole buffer at every step, reading the logits at the position of the last real token. The causal mask already guarantees that later positions cannot influence an earlier one, so the padding on the right is inert. The RoPE tables and the mask are constants over the loop, so they are built once and passed in rather than rebuilt per token.

One shape, one compile. The cell below reports the result on the machine this notebook was last run on: **4.6 s for the first call including the compile, then 0.2 s** — about 3 ms/token, roughly **20× faster per token** and **90× faster on the first call**. The generated text is *identical*; this changes only how the computation is scheduled, not what it computes.

The same lesson appears wherever JAX meets a loop over variable-length data, and it is the single largest cost in `nanochat-grpo.ipynb`, where rollout sampling dominates training time for exactly this reason.

In [6]:
@jax.jit
def next_token_logits(params, buf, n, cos, sin, mask):
    # buf always has shape (seq_len,) and cos/sin/mask are passed in rather than
    # rebuilt, so XLA compiles this exactly once.
    logits = forward(params, buf[None], cos, sin, mask)
    return logits[0, n - 1]

def generate(model, prompt, max_new_tokens=80, temperature=1.0,
             top_k=0, top_p=1.0, key=None):
    params, model_cfg = model['params'], model['cfg']
    model_vocab, model_merges = model['vocab'], model['merges']
    if key is None:
        key = jax.random.key(0)

    seq_len = model_cfg['seq_len']
    cos, sin = precompute_rope(seq_len, model_cfg['head_dim'])   # hoisted out of the loop
    mask = causal_mask(seq_len)

    ids = bpe_encode(prompt, model_vocab, model_merges)[-seq_len:]
    n = len(ids)
    buf = jnp.zeros(seq_len, dtype=jnp.int32).at[:n].set(jnp.array(ids))

    new_tokens = []
    for _ in range(max_new_tokens):
        if n >= seq_len:                 # context window is full
            break
        logits = next_token_logits(params, buf, n, cos, sin, mask)
        key, subkey = jax.random.split(key)
        token = int(sample_token(subkey, logits, temperature, top_k, top_p))
        new_tokens.append(token)
        buf = buf.at[n].set(token)
        n += 1

    return bpe_decode(new_tokens, model_vocab)

In [7]:
t0 = time.time()
text = generate(demo, prompt, max_new_tokens=60, temperature=0.8, key=jax.random.key(0))
first = time.time() - t0

t0 = time.time()
_ = generate(demo, prompt, max_new_tokens=60, temperature=0.8, key=jax.random.key(1))
second = time.time() - t0

print(f'first call  (compiles once): {first:5.1f} s')
print(f'second call (compiled):      {second:5.1f} s   ->  {second / 60 * 1000:.0f} ms/token')
print()
print(text)

first call  (compiles once):   4.6 s
second call (compiled):        0.2 s   ->  3 ms/token

lived in a small house with a big bed. She loved to eat food and play with her friends. One day, Fluffy found a big bowl of yoat


## 5. What pretraining, SFT and GRPO each bought

This is the point of the notebook: the same prompt and the same sampling key, through all three checkpoints.

The first prompt is the **SFT task** — continue a story, wrapped in the chat template the SFT notebook trained on.

In [8]:
def compare(prompt, max_new_tokens=60, seed=0, **kwargs):
    print(f'PROMPT: {prompt}')
    print('=' * 78)
    for name, model in models.items():
        text = generate(model, prompt, max_new_tokens=max_new_tokens,
                        key=jax.random.key(seed), **kwargs)
        print(f'[{name}]')
        print(text)
        print('-' * 78)

compare('[INST] Continue the story: Once upon a time there was a little cat named Mia. [/INST] ',
        temperature=0.8)

PROMPT: [INST] Continue the story: Once upon a time there was a little cat named Mia. [/INST] 


[pretrained]
lettersang was their favorite drawer and she was always sad. 
Tony hugged Tiny and said, "It's okay. I know we can. I 
------------------------------------------------------------------------------


[SFT]
lived in a small house with a big bed. She loved to eat food and play with her friends. One day, Fluffy found a big bowl of yoat
------------------------------------------------------------------------------


[GRPO]
lived in a small house with a big bathrobe. It was red and shiny, and she loved to sleep on the soft bed. She named Fluff
------------------------------------------------------------------------------


The pretrained model has never seen `[INST]` as anything but ordinary characters. It was trained to continue TinyStories text, so it continues — but it continues the *whole prompt as a fragment of a story*, drifting into unrelated characters rather than picking up where Mia left off. Nothing is wrong with it; it is doing precisely what it was trained to do, which happens not to be what we asked.

The SFT model has learned the convention: after `[/INST]`, produce a continuation of the sentence that came after `Continue the story:`. That is the whole value of supervised fine-tuning here — not new knowledge about language, but knowing which span of text it is expected to produce.

The GRPO model was initialised from the SFT model, so it keeps the format and differs only where its reward pushed it.

The second prompt is the **GRPO task**, which SFT was never trained on: a story about an animal, in a specified number of sentences.

In [9]:
compare('[INST] Write a story about a fox in exactly 3 sentences. [/INST]', temperature=0.8)

PROMPT: [INST] Write a story about a fox in exactly 3 sentences. [/INST]


[pretrained]
Doz'santa had gone be forevering a little boy."Smoke and was brave and they had a great time playing outside. Steve this tup
------------------------------------------------------------------------------


[SFT]
y, Sarah, and her friends invited the tomato go together, and they also could see the fun view. Mom was so excited to see all
------------------------------------------------------------------------------


[GRPO]
flea. Everyone was so excited he was a very clever boy. And he loved watching his favorite TV as it went. He would follow 
------------------------------------------------------------------------------


### Counting, rather than squinting

Judging "exactly three sentences" by eye over three samples is not evidence. The GRPO reward counted sentences mechanically, so we can apply the same measurement over enough samples to mean something.

Two honest caveats about the numbers below. Generation stops after `max_new_tokens`, so a story that would have run to four sentences can be truncated mid-sentence and scored as three — the metric flatters every model, and the GRPO model learned partly to exploit exactly this. And 20 samples per model gives a standard error of roughly 0.1, so only large gaps here are real.

In [10]:
def count_sentences(text):
    return len([s for s in re.split(r'[.!?]+', text) if s.strip()])

target = 3
counting_prompt = f'[INST] Write a story about a fox in exactly {target} sentences. [/INST]'
n_samples = 20

print(f'samples with exactly {target} sentences, out of {n_samples}')
for name, model in models.items():
    hits = 0
    for i in range(n_samples):
        text = generate(model, counting_prompt, max_new_tokens=80,
                        temperature=0.9, key=jax.random.key(1000 + i))
        hits += (count_sentences(text) == target)
    print(f'  {name:11s} {hits:2d}/{n_samples}   ({hits / n_samples:.0%})')

samples with exactly 3 sentences, out of 20


  pretrained   4/20   (20%)


  SFT          9/20   (45%)


  GRPO        12/20   (60%)


## 6. The sampling knobs, side by side

One model, one prompt, one random key — only the decoding rule changes. Greedy is the instructive one: watch it commit to safe, generic phrasing and start circling.

In [11]:
settings = [
    ('greedy',           dict(temperature=0.0)),
    ('T=0.5',            dict(temperature=0.5)),
    ('T=1.0',            dict(temperature=1.0)),
    ('T=1.2',            dict(temperature=1.2)),
    ('T=1.0, top_k=10',  dict(temperature=1.0, top_k=10)),
    ('T=1.0, top_p=0.9', dict(temperature=1.0, top_p=0.9)),
]

for label, kwargs in settings:
    text = generate(demo, prompt, max_new_tokens=40, key=jax.random.key(7), **kwargs)
    print(f'{label:17s} {text}')
    print()

greedy            was very excited because she had a big smile on her face. She loved to play with her friends in

T=0.5             was a happy cat who liked to play with his ball. One day, Fluffy went outside to play with her



T=1.0             was a brave cat who lived in a big room. She liked to nap in the sun all day 

T=1.2             was a brave cat who laughed was a cat who found from a big, warm blanket and couldn't help



T=1.0, top_k=10   was a bit filthy because he was so thirsty. He wanted to go outside, especi



T=1.0, top_p=0.9  was a brave cat who lived in a big house. She loved to catch the house in her little 



## 7. An interactive chat loop

Everything above is one function call away from a chat interface. The loop reads a line, wraps it in the template the model was fine-tuned on, generates, and prints.

The cell below only *defines* `chat` — calling it blocks on `input()`, which would hang a non-interactive run of the notebook. Uncomment the last line to talk to the model.

In [12]:
def chat(model, temperature=0.8, top_p=0.95, max_new_tokens=100, seed=42):
    # Read-generate loop. Type 'quit', or interrupt the kernel, to stop.
    key = jax.random.key(seed)
    print("Type a message. 'quit' to stop.")
    while True:
        try:
            user = input('You: ').strip()
        except (EOFError, KeyboardInterrupt):
            print()
            break
        if not user or user.lower() in ('quit', 'exit'):
            break
        key, subkey = jax.random.split(key)
        reply = generate(model, f'[INST] {user} [/INST] ',
                         max_new_tokens=max_new_tokens, temperature=temperature,
                         top_p=top_p, key=subkey)
        print(f'Model: {reply.strip()}')
        print()

# chat(models.get('GRPO', reference))

## 8. What is missing

The generation loop above is honest but minimal. Four things a production implementation would add, each of which is visible in the outputs:

**No KV cache.** Every step recomputes the keys and values for the entire buffer, including the hundreds of positions that have not changed since the previous token. Caching them makes generation roughly another 5× faster and turns the per-token cost from linear in context length to constant. It is the obvious next optimisation after the shape fix in section 4.

**No end-of-text token.** The vocabulary has no way to spell "I am finished", so the model cannot stop and we simply cut it off after `max_new_tokens`. That is why the samples above so often end mid-word, and why the sentence-counting metric is biased. Adding an `<|endoftext|>` token means a new vocabulary entry, re-tokenising the corpus and retraining — not a one-line fix.

**A 256-token context.** `seq_len` is 256, so the buffer fills after a couple of exchanges and `generate` stops. A real chat loop has to decide what to evict.

**The chat template is a convention, not a structure.** `[INST]` is ordinary text that BPE shreds into several tokens; nothing prevents the model from emitting it mid-reply, and nothing prevents a user from typing it to impersonate the template. Real chat models use reserved special tokens precisely so that the boundary between user and assistant cannot be forged from user text — the same class of problem as the prompt injection in `minisweagent.ipynb`, which is where we go next.

## Exercises

1. **Greedy loops.** Generate 200 tokens with `temperature=0.0`. How long before the output starts repeating? Now try `temperature=0.01`. Explain why such a tiny amount of noise changes the behaviour so much.

2. **Top-k against top-p.** Using the distribution from section 3, find a prompt where the model is very confident and one where it is not. For each, report how many tokens `top_p=0.9` keeps. Then argue which of `top_k=10` and `top_p=0.9` you would ship, and why.

3. **Is the GRPO gap real?** Section 5 measures 20 samples per model. Estimate the standard error of those fractions, then raise `n_samples` until the ordering is stable. Does the GRPO model still beat the SFT model? Separately, re-run with `max_new_tokens=200` — how much of the original gap was truncation?

4. **KV cache.** Implement one. Keep per-layer `K` and `V` buffers of shape `(H, seq_len, head_dim)`, write the new token's projections at position `n`, and attend over `[:n+1]`. Confirm that greedy decoding produces *exactly* the same tokens as the current implementation, then measure the speedup.

5. **Stopping.** Without retraining, add a stop condition to `generate`: end as soon as the decoded text contains `target` sentence terminators. Re-run the measurement in section 5. How much of the difference between the three models survives?

# Colophon
This notebook was written by [Yoav Ram](http://python.yoavram.com).

This work is licensed under a [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) International License.

![Python logo](https://www.python.org/static/community_logos/python-logo.png)